In [1]:

# ============================================================
# SPATIOTEMPORAL GNN PROJECT
# GRU BASELINE: VALIDATION-BASED HYPERPARAMETER SEARCH
# ============================================================

import os
import glob
import json
import time
import copy
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 1. REPRODUCIBILITY AND DEVICE
# ============================================================

BASE_SEED = 42

torch.manual_seed(BASE_SEED)
np.random.seed(BASE_SEED)
random.seed(BASE_SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(BASE_SEED)
else:
    device = torch.device("cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("GPU not detected. Training will use CPU.")


# ============================================================
# 2. FIND AND EXTRACT THE KAGGLE DATASET
# ============================================================

INPUT_DIR = Path("/kaggle/input/datasets/drrajgauravmishra/spatiotemporal-gnn-kaggle-data")
WORK_DIR = Path("/kaggle/working")

RESULTS_DIR = WORK_DIR / "gru_baseline_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# First, look for the NPZ files directly.
training_files = list(INPUT_DIR.rglob("training_annual.npz"))

# If the uploaded dataset contains a ZIP archive,
# extract it into the working directory.
if not training_files:

    zip_files = list(INPUT_DIR.rglob("*.zip"))

    if not zip_files:
        raise FileNotFoundError(
            "Could not find training_annual.npz or a ZIP archive "
            "inside /kaggle/input. Please check the attached dataset."
        )

    extraction_dir = WORK_DIR / "extracted_dataset"
    extraction_dir.mkdir(parents=True, exist_ok=True)

    for zip_file in zip_files:
        print("Extracting:", zip_file.name)

        with zipfile.ZipFile(zip_file, "r") as zip_ref:
            zip_ref.extractall(extraction_dir)

    training_files = list(
        extraction_dir.rglob("training_annual.npz")
    )

if not training_files:
    raise FileNotFoundError(
        "training_annual.npz was not found after extraction."
    )

DATA_DIR = training_files[0].parent

print("\nDataset directory:", DATA_DIR)

required_files = [
    "training_annual.npz",
    "validation_annual.npz",
    "testing_annual.npz",
    "annual_forecasting_metadata.json"
]

for filename in required_files:
    file_path = DATA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )

    print("Found:", filename)


# ============================================================
# 3. LOAD DATA
# ============================================================

train_data = np.load(
    DATA_DIR / "training_annual.npz",
    allow_pickle=False
)

val_data = np.load(
    DATA_DIR / "validation_annual.npz",
    allow_pickle=False
)

test_data = np.load(
    DATA_DIR / "testing_annual.npz",
    allow_pickle=False
)

with open(
    DATA_DIR / "annual_forecasting_metadata.json",
    "r"
) as f:
    metadata = json.load(f)


# NPZ structure:
# X, Y, Y_mask, input_indices, target_indices, target_years

X_train = train_data["X"].astype(np.float32)
Y_train = train_data["Y"].astype(np.float32)
Y_train_mask = train_data["Y_mask"].astype(bool)

X_val = val_data["X"].astype(np.float32)
Y_val = val_data["Y"].astype(np.float32)
Y_val_mask = val_data["Y_mask"].astype(bool)

X_test = test_data["X"].astype(np.float32)
Y_test = test_data["Y"].astype(np.float32)
Y_test_mask = test_data["Y_mask"].astype(bool)

print("\nDATASET SHAPES")
print("-" * 50)

print("Training X:", X_train.shape)
print("Training Y:", Y_train.shape)
print("Training mask:", Y_train_mask.shape)

print("Validation X:", X_val.shape)
print("Validation Y:", Y_val.shape)

print("Testing X:", X_test.shape)
print("Testing Y:", Y_test.shape)

print("\nTraining target years:",
      train_data["target_years"])

print("Validation target years:",
      val_data["target_years"])

print("Testing target years:",
      test_data["target_years"])


# ============================================================
# 4. DATA INTEGRITY CHECKS
# ============================================================

assert X_train.shape[1:] == X_val.shape[1:]
assert X_train.shape[1:] == X_test.shape[1:]

assert Y_train.shape == Y_train_mask.shape
assert Y_val.shape == Y_val_mask.shape
assert Y_test.shape == Y_test_mask.shape

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

assert np.array_equal(Y_train_mask, np.isfinite(Y_train))
assert np.array_equal(Y_val_mask, np.isfinite(Y_val))
assert np.array_equal(Y_test_mask, np.isfinite(Y_test))

print("\nData integrity checks passed.")


# ============================================================
# 5. TARGET NORMALIZATION
# Training data only; validation and testing are not used.
# ============================================================

target_names = ["Rainfall", "Maximum Temperature"]

target_mean = np.zeros(2, dtype=np.float32)
target_std = np.zeros(2, dtype=np.float32)

for j in range(2):

    valid_values = Y_train[:, :, j][
        Y_train_mask[:, :, j]
    ]

    target_mean[j] = valid_values.mean()
    target_std[j] = valid_values.std()

    if target_std[j] == 0:
        target_std[j] = 1.0

print("\nTRAINING-ONLY NORMALIZATION")
print("-" * 50)

for j, name in enumerate(target_names):
    print(
        f"{name}: mean={target_mean[j]:.6f}, "
        f"std={target_std[j]:.6f}"
    )


def normalize_targets(Y, mask):

    Y_norm = np.zeros_like(Y, dtype=np.float32)

    for j in range(2):

        valid = mask[:, :, j]

        Y_norm[:, :, j][valid] = (
            Y[:, :, j][valid] - target_mean[j]
        ) / target_std[j]

    return Y_norm


Y_train_norm = normalize_targets(
    Y_train, Y_train_mask
)

Y_val_norm = normalize_targets(
    Y_val, Y_val_mask
)

Y_test_norm = normalize_targets(
    Y_test, Y_test_mask
)


# ============================================================
# 6. CONVERT ARRAYS TO PYTORCH TENSORS
# ============================================================

X_train_t = torch.from_numpy(X_train).to(device)
Y_train_t = torch.from_numpy(Y_train_norm).to(device)
Y_train_mask_t = torch.from_numpy(Y_train_mask).to(device)

X_val_t = torch.from_numpy(X_val).to(device)
Y_val_t = torch.from_numpy(Y_val_norm).to(device)
Y_val_mask_t = torch.from_numpy(Y_val_mask).to(device)

X_test_t = torch.from_numpy(X_test).to(device)
Y_test_mask_t = torch.from_numpy(Y_test_mask).to(device)

print("\nTensor preparation completed.")


# ============================================================
# 7. MASKED MSE LOSS
# ============================================================

def masked_mse_loss(predictions, targets, mask):

    valid_predictions = predictions[mask]
    valid_targets = targets[mask]

    return torch.mean(
        (valid_predictions - valid_targets) ** 2
    )


# ============================================================
# 8. GRU BASELINE ARCHITECTURE
# ============================================================

class GRUBaseline(nn.Module):

    def __init__(
        self,
        input_dim=4,
        hidden_dim=32,
        output_dim=2,
        dropout=0.2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        batch_size, time_steps, num_nodes, num_features = x.shape

        # Process each spatial node independently.
        x = x.permute(0, 2, 1, 3)

        x = x.reshape(
            batch_size * num_nodes,
            time_steps,
            num_features
        )

        _, hidden = self.gru(x)

        representation = self.dropout(hidden[-1])

        output = self.fc(representation)

        return output.reshape(
            batch_size,
            num_nodes,
            2
        )


# ============================================================
# 9. GRU TRAINING FUNCTION
# ============================================================

def train_gru_model(
    model,
    X_train,
    Y_train_norm,
    Y_train_mask,
    X_val,
    Y_val_norm,
    Y_val_mask,
    max_epochs,
    learning_rate,
    weight_decay,
    patience,
    seed
):

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    history = {
        "train_loss": [],
        "val_loss": []
    }

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(max_epochs):

        # Training
        model.train()

        optimizer.zero_grad()

        train_predictions = model(X_train)

        train_loss = masked_mse_loss(
            train_predictions,
            Y_train_norm,
            Y_train_mask
        )

        train_loss.backward()

        optimizer.step()

        # Validation
        model.eval()

        with torch.no_grad():

            val_predictions = model(X_val)

            val_loss = masked_mse_loss(
                val_predictions,
                Y_val_norm,
                Y_val_mask
            )

        train_loss_value = train_loss.item()
        val_loss_value = val_loss.item()

        history["train_loss"].append(
            train_loss_value
        )

        history["val_loss"].append(
            val_loss_value
        )

        # Best model selection
        if val_loss_value < best_val_loss:

            best_val_loss = val_loss_value
            best_epoch = epoch + 1

            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        # Early stopping
        if epochs_without_improvement >= patience:
            break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed_time = time.time() - start_time

    return {
        "model": model,
        "state_dict": best_state,
        "history": history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_trained": len(history["train_loss"]),
        "training_time_seconds": elapsed_time
    }


# ============================================================
# 10. GRU HYPERPARAMETER CONFIGURATION
# ============================================================

GRU_CONFIG = {
    "hidden_sizes": [64, 128],
    "epoch_options": [300, 500],
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    "patience": 30,
    "seeds": [42, 123, 2023],
    "training_mode": "full_batch"
}

print("\nGRU CONFIGURATION")
print("-" * 50)

for key, value in GRU_CONFIG.items():
    print(f"{key}: {value}")


# ============================================================
# 11. VALIDATION-BASED HYPERPARAMETER SEARCH
# ============================================================

gru_search_results = []
gru_model_states = {}

total_runs = (
    len(GRU_CONFIG["hidden_sizes"])
    * len(GRU_CONFIG["epoch_options"])
    * len(GRU_CONFIG["seeds"])
)

run_number = 0

search_start_time = time.time()

for hidden_size in GRU_CONFIG["hidden_sizes"]:

    for max_epochs in GRU_CONFIG["epoch_options"]:

        for seed in GRU_CONFIG["seeds"]:

            run_number += 1

            print("\n" + "=" * 60)

            print(
                f"Run {run_number}/{total_runs} | "
                f"Hidden size: {hidden_size} | "
                f"Max epochs: {max_epochs} | "
                f"Seed: {seed}"
            )

            # Set seeds before model initialization
            torch.manual_seed(seed)
            np.random.seed(seed)
            random.seed(seed)

            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            model = GRUBaseline(
                input_dim=4,
                hidden_dim=hidden_size,
                output_dim=2,
                dropout=0.2
            ).to(device)

            result = train_gru_model(
                model=model,
                X_train=X_train_t,
                Y_train_norm=Y_train_t,
                Y_train_mask=Y_train_mask_t,
                X_val=X_val_t,
                Y_val_norm=Y_val_t,
                Y_val_mask=Y_val_mask_t,
                max_epochs=max_epochs,
                learning_rate=GRU_CONFIG["learning_rate"],
                weight_decay=GRU_CONFIG["weight_decay"],
                patience=GRU_CONFIG["patience"],
                seed=seed
            )

            gru_search_results.append({

                "hidden_size": hidden_size,
                "max_epochs": max_epochs,
                "seed": seed,

                "best_epoch": result["best_epoch"],
                "epochs_trained": result["epochs_trained"],

                "best_val_loss": result["best_val_loss"],

                "training_time_seconds":
                    result["training_time_seconds"]
            })

            # Store best checkpoint for later test evaluation
            checkpoint_key = (
                hidden_size,
                max_epochs,
                seed
            )

            gru_model_states[checkpoint_key] = (
                result["state_dict"]
            )

            print(
                f"Best validation loss: "
                f"{result['best_val_loss']:.6f}"
            )

            print(
                f"Best epoch: {result['best_epoch']}"
            )

            print(
                f"Epochs trained: {result['epochs_trained']}"
            )

            print(
                f"Training time: "
                f"{result['training_time_seconds']:.2f} seconds"
            )

            del model, result

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


search_total_time = time.time() - search_start_time

gru_search_df = pd.DataFrame(gru_search_results)

print("\n" + "=" * 60)
print("GRU HYPERPARAMETER SEARCH COMPLETED")
print("Total runs:", len(gru_search_df))
print("Total time (minutes):",
      round(search_total_time / 60, 2))


# ============================================================
# 12. AGGREGATE RESULTS ACROSS RANDOM SEEDS
# ============================================================

gru_summary = (
    gru_search_df
    .groupby(["hidden_size", "max_epochs"])
    .agg(
        mean_validation_loss=("best_val_loss", "mean"),
        std_validation_loss=("best_val_loss", "std"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_training_time=("training_time_seconds", "mean")
    )
    .reset_index()
)

gru_summary = gru_summary.sort_values(
    "mean_validation_loss"
).reset_index(drop=True)

print("\nVALIDATION-BASED CONFIGURATION SUMMARY")
print("-" * 70)

print(
    gru_summary.to_string(index=False)
)


# ============================================================
# 13. SELECT BEST CONFIGURATION
# Selection uses validation loss only.
# ============================================================

best_configuration = gru_summary.iloc[0]

best_hidden_size = int(
    best_configuration["hidden_size"]
)

best_max_epochs = int(
    best_configuration["max_epochs"]
)

print("\nSELECTED GRU CONFIGURATION")
print("-" * 50)

print("Hidden size:", best_hidden_size)
print("Maximum epochs:", best_max_epochs)

print(
    "Mean validation loss:",
    round(
        best_configuration["mean_validation_loss"],
        6
    )
)

print(
    "Standard deviation:",
    round(
        best_configuration["std_validation_loss"],
        6
    )
)


# ============================================================
# 14. FINAL TEST EVALUATION
# Test set is used only after configuration selection.
# ============================================================

def calculate_metrics(y_true, y_pred, mask):

    results = {}

    for j, name in enumerate(target_names):

        valid = mask[:, :, j]

        actual = y_true[:, :, j][valid]
        predicted = y_pred[:, :, j][valid]

        mae = np.mean(
            np.abs(actual - predicted)
        )

        rmse = np.sqrt(
            np.mean((actual - predicted) ** 2)
        )

        ss_res = np.sum(
            (actual - predicted) ** 2
        )

        ss_tot = np.sum(
            (actual - actual.mean()) ** 2
        )

        r2 = (
            1 - ss_res / ss_tot
            if ss_tot > 0
            else np.nan
        )

        results[name] = {
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "valid_count": len(actual)
        }

    return results


gru_test_results = []

selected_seed_states = {}

for seed in GRU_CONFIG["seeds"]:

    print(f"\nEvaluating selected configuration | Seed: {seed}")

    checkpoint_key = (
        best_hidden_size,
        best_max_epochs,
        seed
    )

    model = GRUBaseline(
        input_dim=4,
        hidden_dim=best_hidden_size,
        output_dim=2,
        dropout=0.2
    ).to(device)

    model.load_state_dict(
        gru_model_states[checkpoint_key]
    )

    model.eval()

    with torch.no_grad():

        predictions_norm = model(
            X_test_t
        ).cpu().numpy()

    # Inverse normalization
    predictions = (
        predictions_norm * target_std.reshape(1, 1, 2)
        + target_mean.reshape(1, 1, 2)
    )

    metrics = calculate_metrics(
        Y_test,
        predictions,
        Y_test_mask
    )

    for variable, values in metrics.items():

        gru_test_results.append({

            "hidden_size": best_hidden_size,
            "max_epochs": best_max_epochs,
            "seed": seed,
            "variable": variable,

            "MAE": values["MAE"],
            "RMSE": values["RMSE"],
            "R2": values["R2"],

            "valid_count": values["valid_count"]
        })

    # Save the selected model checkpoint
    selected_seed_states[seed] = model.state_dict()

    del model


gru_test_df = pd.DataFrame(
    gru_test_results
)

print("\nFINAL GRU TEST RESULTS")
print("-" * 70)

print(
    gru_test_df.to_string(index=False)
)


# ============================================================
# 15. TEST RESULTS: MEAN AND STANDARD DEVIATION
# ============================================================

gru_test_summary = (
    gru_test_df
    .groupby("variable")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std")
    )
    .reset_index()
)

print("\nTEST PERFORMANCE: MEAN ± STANDARD DEVIATION")
print("-" * 70)

print(
    gru_test_summary.to_string(index=False)
)


# ============================================================
# 16. SAVE EXPERIMENTAL RESULTS
# ============================================================

gru_search_df.to_csv(
    RESULTS_DIR / "gru_hyperparameter_search.csv",
    index=False
)

gru_summary.to_csv(
    RESULTS_DIR / "gru_validation_summary.csv",
    index=False
)

gru_test_df.to_csv(
    RESULTS_DIR / "gru_test_results.csv",
    index=False
)

gru_test_summary.to_csv(
    RESULTS_DIR / "gru_test_summary.csv",
    index=False
)

# Save selected configuration
selected_config = {
    "model": "GRU Baseline",
    "hidden_size": best_hidden_size,
    "max_epochs": best_max_epochs,
    "learning_rate": GRU_CONFIG["learning_rate"],
    "weight_decay": GRU_CONFIG["weight_decay"],
    "patience": GRU_CONFIG["patience"],
    "seeds": GRU_CONFIG["seeds"],
    "training_mode": GRU_CONFIG["training_mode"],
    "selection_metric": "Mean validation masked MSE",
    "target_mean": target_mean.tolist(),
    "target_std": target_std.tolist()
}

with open(
    RESULTS_DIR / "gru_selected_configuration.json",
    "w"
) as f:

    json.dump(
        selected_config,
        f,
        indent=4
    )

# Save selected model checkpoints
for seed, state_dict in selected_seed_states.items():

    torch.save(
        state_dict,
        RESULTS_DIR / f"gru_best_seed_{seed}.pt"
    )

print("\nRESULTS SAVED")
print("-" * 50)

for file_path in sorted(RESULTS_DIR.iterdir()):
    print(
        file_path.name,
        "|",
        round(file_path.stat().st_size / 1024, 2),
        "KB"
    )

print("\nAll GRU experiments completed successfully.")


Device: cuda
GPU: Tesla T4
GPU memory (GB): 15.64

Dataset directory: /kaggle/input/datasets/drrajgauravmishra/spatiotemporal-gnn-kaggle-data
Found: training_annual.npz
Found: validation_annual.npz
Found: testing_annual.npz
Found: annual_forecasting_metadata.json

DATASET SHAPES
--------------------------------------------------
Training X: (8, 10, 5040, 4)
Training Y: (8, 5040, 2)
Training mask: (8, 5040, 2)
Validation X: (6, 10, 5040, 4)
Validation Y: (6, 5040, 2)
Testing X: (2, 10, 5040, 4)
Testing Y: (2, 5040, 2)

Training target years: [2010 2011 2012 2013 2014 2015 2016 2017]
Validation target years: [2018 2019 2020 2021 2022 2023]
Testing target years: [2024 2025]

Data integrity checks passed.

TRAINING-ONLY NORMALIZATION
--------------------------------------------------
Rainfall: mean=3.058888, std=2.156659
Maximum Temperature: mean=31.361950, std=2.990062

Tensor preparation completed.

GRU CONFIGURATION
--------------------------------------------------
hidden_sizes: [64, 1